# merimee-rag — évaluation sur GPU (Google Colab)

**Avant de commencer**
1. Menu *Exécution → Modifier le type d'exécution → GPU T4*.
2. Dépose `merimee-rag-colab.zip` dans ton Google Drive, dans un dossier **`merimee-rag-colab`**.

**Si la session se coupe** (inactivité, quota GPU) : relance les cellules 1 à 4, puis l'étape où tu en étais.
Le cache des réponses du LLM et les résultats sont enregistrés dans le Drive au fur et à mesure :
rien n'est perdu, ce qui a déjà été calculé ressort instantanément.

In [ ]:
# 1. Vérifier le GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Google Drive + copie du projet sur le disque local de Colab (plus rapide que de travailler dans le Drive)
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess
DRIVE = '/content/drive/MyDrive/merimee-rag-colab'
PROJ = '/content/merimee-rag'
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists(PROJ):
    import glob
    zips = glob.glob(f'{DRIVE}/**/merimee-rag-colab.zip', recursive=True)
    dossiers = [d for d in glob.glob(f'{DRIVE}/**/merimee-rag', recursive=True) if os.path.exists(f'{d}/pyproject.toml')]
    if zips:                       # zip déposé tel quel
        !unzip -q "{zips[0]}" -d /content
    elif dossiers:                 # zip déjà décompressé par Windows avant l'envoi
        print('Copie depuis', dossiers[0])
        shutil.copytree(dossiers[0], PROJ, ignore=shutil.ignore_patterns('.venv', '__pycache__'))
    else:
        raise FileNotFoundError(f'Ni zip ni dossier merimee-rag trouvé dans {DRIVE}')
os.chdir(PROJ)

# Tout ce qui doit survivre à une coupure vit dans le Drive
os.environ['MERIMEE_LLM_CACHE'] = f'{DRIVE}/llm_cache.jsonl'
os.makedirs(f'{DRIVE}/results', exist_ok=True)
if os.path.exists(f'{DRIVE}/testset.jsonl'):          # jeu de test déjà généré lors d'une session précédente
    shutil.copy(f'{DRIVE}/testset.jsonl', 'eval/testset.jsonl')

def sauver():
    """Copie jeu de test et résultats dans le Drive."""
    if os.path.exists('eval/testset.jsonl'):
        shutil.copy('eval/testset.jsonl', f'{DRIVE}/testset.jsonl')
    if os.path.exists('results'):
        shutil.copytree('results', f'{DRIVE}/results', dirs_exist_ok=True)
    print('✔ sauvegardé dans', DRIVE)

print(os.listdir('data/index'))

In [ ]:
# 3. Dépendances Python
!pip install -q rank_bm25 snowballstemmer sentence-transformers
!pip install -q -e . --no-deps

In [ ]:
# 4. Ollama sur le GPU + modèles
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

env = dict(os.environ, OLLAMA_MAX_LOADED_MODELS='2', OLLAMA_KEEP_ALIVE='2h')  # 3B et 7B restent chargés ensemble
ollama = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
import time; time.sleep(5)
!ollama pull qwen2.5:3b
!ollama pull qwen2.5:7b
!ollama list

### Contrôle rapide (≈ 1 min) — doit afficher 8/8 ou 7/8

In [ ]:
!python -m eval.smoke --model qwen2.5:3b

### Étape A — Créer le jeu de test
Le 7B lit ~150 notices tirées au hasard (réparties par région) et écrit pour chacune une question, la réponse
et la phrase exacte qui la justifie. Les paires dont la phrase n'existe pas dans la notice sont jetées.

In [ ]:
if not os.path.exists('eval/testset.jsonl'):
    !python -m eval.build_testset --n 150
else:
    print('Jeu de test déjà présent (session précédente) :', sum(1 for _ in open('eval/testset.jsonl')), 'questions')
sauver()

In [ ]:
# Aperçu de quelques questions générées : vérifie à l'œil qu'elles sont sensées
import json, random
qs = [json.loads(l) for l in open('eval/testset.jsonl')]
for q in random.Random(0).sample([q for q in qs if q['type'] == 'synthetic'], 5):
    print('Q :', q['question']); print('R :', q['answer']); print('   ↳', q['evidence'][:150], '\n')

### Étape B — Évaluer la recherche (BM25 / dense / hybride)

In [ ]:
!python -m eval.eval_retrieval
sauver()

### Étape C — Évaluer les réponses
4 configurations (sans RAG, BM25, dense, hybride) × 50 questions + 15 hors corpus, jugées par le 7B.
Commence par `--limit 20` si tu veux un premier aperçu ; relancer ensuite avec `--limit 50` réutilise le cache.

In [ ]:
!python -m eval.eval_generation --limit 50
sauver()

### Étape D — Rapport

In [ ]:
!python -m eval.report
sauver()
from IPython.display import Markdown, Image, display
display(Markdown(open('results/REPORT.md', encoding='utf-8').read().replace('](', '](results/')))
for f in ('results/retrieval.png', 'results/generation.png'):
    if os.path.exists(f): display(Image(f))

In [ ]:
# Télécharger les résultats (à déposer dans le dossier results/ du projet sur ton PC)
!cd /content/merimee-rag && zip -qr /content/resultats.zip results eval/testset.jsonl
from google.colab import files
files.download('/content/resultats.zip')